# Task 9 — Clean Texture Pilot (Stage D)

Thin Colab launcher only. All model and training code lives in `src/cya_detector/models/texture.py` and `src/cya_detector/training/texture_stage_d.py`; this notebook invokes it only through `scripts/extract_texture_features.py`, `scripts/train_texture_pilot.py`, and `scripts/compare_texture_pilot.py` (the same CLIs wired to the `task9-*` Make targets).

**Clean-only boundary.** This pilot trains and selects only on the fixed-Q96 matched-clean manifest's `seed_train` (training) and `selection_val` (comparison) splits. It never reads `self_train_pool`, sealed `final_test`, source-original images, Task 3 robustness variants, or Task 8B data. CLIP stays fully frozen throughout; only the lightweight Task 9 heads train.

**Fixed experiment contract.** Up to four non-overlapping 112x112 source patches per image (the locked CLIP processor upsamples each to 336x336), for a maximum encoding budget of one global view plus four local views (five views total) per image. Three variants (`global_only`, `local_only`, `global_local`) each train with seeds 42, 43, and 44 — nine runs total from the same frozen-feature caches extracted once.

**Clean gate.** Passing the gate (decision `continue_to_robustness_design`) only authorizes a later, separately specified Task 3 robustness continuation; it does not by itself retain Task 9. A `reject_texture_clean_gate` decision means Task 9 remains tested-and-rejected and RINE (Task 6) remains the retained global representation. This notebook only wires up the launcher/commands — it does not itself assert a retention decision.

In [ ]:
# 1. GPU and Drive.
import torch
assert torch.cuda.is_available(), 'Task 9 requires a GPU Colab server'
print('GPU:', torch.cuda.get_device_name(0))
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Refresh repository and dependencies.
from pathlib import Path
import subprocess
import sys
import shutil

PROJECT_ROOT = Path('/content/cya-techjam26')
REPOSITORY_URL = 'https://github.com/maxi-cmyk/cya-techjam26.git'
if (PROJECT_ROOT / '.git').is_dir():
    status = subprocess.run(['git', 'status', '--porcelain'], cwd=PROJECT_ROOT, check=True, capture_output=True, text=True).stdout.splitlines()
    unexpected = [line for line in status if not line.endswith('configs/colab.json')]
    assert not unexpected, f'Unexpected remote checkout changes: {unexpected}'
    if status:
        subprocess.run(['git', 'restore', 'configs/colab.json'], cwd=PROJECT_ROOT, check=True)
    subprocess.run(['git', 'pull', '--ff-only'], cwd=PROJECT_ROOT, check=True)
else:
    subprocess.run(['git', 'clone', REPOSITORY_URL, str(PROJECT_ROOT)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements-colab.txt'], cwd=PROJECT_ROOT, check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.', '--no-deps'], cwd=PROJECT_ROOT, check=True)

In [ ]:
# 3. Stage the fixed-Q96 manifest and matched-clean images locally.
ARTIFACT_ROOT = PROJECT_ROOT / 'artifacts'
DRIVE_ARTIFACT_ROOT = Path('/content/drive/MyDrive/cya-techjam26/artifacts')
TASK2_ROOT = ARTIFACT_ROOT / 'task2'
input_archive = DRIVE_ARTIFACT_ROOT / 'task2_stagea_bundle.tar.gz'
assert input_archive.is_file(), input_archive
TASK2_ROOT.mkdir(parents=True, exist_ok=True)
shutil.unpack_archive(input_archive, TASK2_ROOT)
manifest = TASK2_ROOT / 'fixed_q96_manifest.csv'
assert manifest.is_file(), manifest
print('Task 9 manifest ready:', manifest)

In [ ]:
# 4. Pin the exact model commit and print the frozen Task 9 contract.
import json
from huggingface_hub import model_info
config_path = PROJECT_ROOT / 'configs/colab.json'
config = json.loads(config_path.read_text())
resolved_commit = model_info(config['model']['identifier'], revision=config['model']['revision']).sha
config['model']['revision'] = resolved_commit
config_path.write_text(json.dumps(config, indent=2) + '\n')
print('Commit:', resolved_commit)
print('Texture contract:', json.dumps(config['texture'], indent=2))

In [ ]:
# 5. Extract and cache frozen global/patch features once; every one of the
#    nine runs below reuses these cached tensors without rerunning CLIP.
import os
GLOBAL_CACHE = Path('/content/rine_feature_cache')
PATCH_CACHE = Path('/content/texture_patch_cache')
CACHE_PAYLOAD = PATCH_CACHE / 'task9_cache_payload.json'
subprocess.run([
    sys.executable, 'scripts/extract_texture_features.py',
    '--manifest', str(manifest),
    '--cache-payload', str(CACHE_PAYLOAD),
    '--global-cache-root', str(GLOBAL_CACHE),
    '--patch-cache-root', str(PATCH_CACHE),
    '--physical-batch-size', '4',
], cwd=PROJECT_ROOT, check=True, env={**os.environ, 'PYTHONUNBUFFERED': '1'})
print('Cache payload ready:', CACHE_PAYLOAD)

In [ ]:
# 6. Resumable per-variant/seed trainer. A run is only copied to Drive after
#    it completes, so no empty Drive directory is created ahead of a result.
#    Completeness requires every artifact the clean gate itself requires, plus
#    a completed status matching this variant/seed, so a run interrupted
#    mid-Drive-copy (leaving metadata present but another artifact missing)
#    is correctly retrained rather than skipped.
TEXTURE_OUTPUT_ROOT = ARTIFACT_ROOT / 'task9'
DRIVE_TEXTURE_ROOT = DRIVE_ARTIFACT_ROOT / 'task9'
EXPERIMENT_NAME = config['texture']['experiment_name']
variants = tuple(config['texture']['variants'])
seeds = tuple(config['texture']['seeds'])

REQUIRED_RUN_ARTIFACTS = (
    'checkpoints/best_clean.pt', 'checkpoints/latest.pt', 'predictions/selection_val.csv',
    'reports/metrics.json', 'reports/training_history.json', 'metadata/run_metadata.json',
)

def run_is_complete(run_root, variant, seed):
    if not all((run_root / relative).is_file() for relative in REQUIRED_RUN_ARTIFACTS):
        return False
    try:
        metadata = json.loads((run_root / 'metadata/run_metadata.json').read_text())
    except (OSError, json.JSONDecodeError):
        return False
    return metadata.get('status') == 'completed' and metadata.get('variant') == variant and metadata.get('seed') == seed

def run_texture(variant, seed):
    local_run = TEXTURE_OUTPUT_ROOT / EXPERIMENT_NAME / variant / f'seed_{seed}'
    drive_run = DRIVE_TEXTURE_ROOT / EXPERIMENT_NAME / variant / f'seed_{seed}'
    if not run_is_complete(local_run, variant, seed) and drive_run.is_dir():
        shutil.copytree(drive_run, local_run, dirs_exist_ok=True)
    if run_is_complete(local_run, variant, seed):
        print(f'SKIP complete: {variant} seed {seed}')
        return
    print(f'RUN texture: {variant} seed {seed}', flush=True)
    subprocess.run([
        sys.executable, 'scripts/train_texture_pilot.py',
        '--cached-features', str(CACHE_PAYLOAD),
        '--variant', variant,
        '--seed', str(seed),
        '--output-root', str(TEXTURE_OUTPUT_ROOT),
        '--device', 'cuda',
        '--overwrite',
    ], cwd=PROJECT_ROOT, check=True, env={**os.environ, 'PYTHONUNBUFFERED': '1'})
    drive_run.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(local_run, drive_run, dirs_exist_ok=True)
    print('SAVED:', drive_run)

In [ ]:
# 7. First proof run; inspect before continuing.
run_texture(variants[0], seeds[0])
first_metrics = json.loads((TEXTURE_OUTPUT_ROOT / EXPERIMENT_NAME / variants[0] / f'seed_{seeds[0]}' / 'reports/metrics.json').read_text())
print(json.dumps(first_metrics, indent=2))

In [ ]:
# 8. Remaining eight runs reuse the shared frozen-feature caches.
for variant in variants:
    for seed in seeds:
        run_texture(variant, seed)

In [ ]:
# 9. Apply the clean gate only after all nine runs are complete. Passing it
#    authorizes a later Task 3 continuation; it does not retain Task 9 alone.
subprocess.run([
    sys.executable, 'scripts/compare_texture_pilot.py',
    '--output-root', str(TEXTURE_OUTPUT_ROOT),
], cwd=PROJECT_ROOT, check=True)
experiment_root = TEXTURE_OUTPUT_ROOT / EXPERIMENT_NAME
for relative in ('comparison', 'metadata'):
    local_dir = experiment_root / relative
    if local_dir.is_dir():
        drive_dir = DRIVE_TEXTURE_ROOT / EXPERIMENT_NAME / relative
        drive_dir.parent.mkdir(parents=True, exist_ok=True)
        shutil.copytree(local_dir, drive_dir, dirs_exist_ok=True)
comparison = json.loads((experiment_root / 'comparison' / 'global_local_comparison.json').read_text())
print('Decision:', comparison['decision'])
print(json.dumps(comparison, indent=2))

## Reading the decision

- `continue_to_robustness_design`: the clean `global_local` candidate beat `global_only` with no more than a 1.0-point per-class regression and at least one corrected error. This authorizes writing a separate Task 3 robustness specification and cost estimate before any transformed Task 9 training input is materialized; it does not retain Task 9 by itself.
- `reject_texture_clean_gate`: stop here. Record Task 9 as tested and rejected; RINE (Task 6) remains the retained global representation, and no transformed Task 9 training should be run.